> [!TIP]
> **Active Development Copy (v2):** This notebook is the workspace for ongoing improvements and extensions to **TBM / ITFM Data Model & Analytical Warehouse**.
> The verified baseline remains preserved in `v1_baseline.ipynb` and `../../02_tbm_itfm_data_model.ipynb`.

# Notebook 02: TBM / ITFM Data Model & Analytical Warehouse

**Series:** Technology Value Intelligence (TBM / ITFM / TVR)  
**Notebook Index:** 02 of 12  
**Focus:** Cost Pools, IT Towers, DuckDB Columnar Warehouse, Materialized Views, Vectorized OLAP  

---

## 1. Business Problem

Traditional corporate accounting records technology spending in General Ledger (GL) accounts (e.g., "Account 5410: Hardware Maintenance", "Account 6120: Professional Services"). While this satisfies statutory accounting requirements, it creates a fundamental blind spot for technology leadership:
- A GL account shows *what type of asset was bought*, but never *what business purpose it serves*.
- The CIO cannot tell from a GL entry whether ₹5 Crores spent on cloud computing supported digital commerce, treasury management, or legacy batch jobs.

To solve this, Technology Business Management (TBM) and IT Financial Management (ITFM) translate financial GL entries into a multi-tier taxonomy:
$$\text{Cost Pools (Financial GL)} \longrightarrow \text{IT Towers / Services} \longrightarrow \text{Applications} \longrightarrow \text{Business Units}$$

---

## 2. Core Concept & Formulation

### 2.1 The TBM-Inspired Multi-Tier Taxonomy
In this implementation, we formalize a simplified TBM/ITFM-inspired model:
1. **Cost Pools:** Elemental financial categories of technology expenditure (Internal Labor, External Labor, Software, Hardware, Telecom, Outside Services, Facilities).
2. **IT Towers / Services:** Standardized technology offerings delivered by IT (e.g., Core Banking, ERP Services, Cloud Platform, Cybersecurity).
3. **Applications:** Software systems deployed to execute IT services.
4. **Business Units:** Organizational consumers accountable for business outcomes.

### 2.2 In-Process Columnar Storage with DuckDB
Rather than deploying heavyweight external database servers (Postgres, Oracle, Snowflake), we utilize **DuckDB** as our local analytical storage engine:
- Vectorized execution engine optimized for analytical aggregations (OLAP).
- Zero network latency and zero memory bloat (runs embedded in-process).
- Seamless zero-copy interoperability with Apache Arrow and pandas DataFrames.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Dynamically locate project root
workspace_root = Path.cwd()
while workspace_root != workspace_root.parent:
    if (workspace_root / "config.yaml").exists() or (workspace_root / "pyproject.toml").exists():
        break
    workspace_root = workspace_root.parent

src_dir = str(workspace_root / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from tvi.config import load_config, get_project_root
from tvi.database import get_database

config = load_config()
db = get_database(config.database.path)

# Ingest all CSVs into DuckDB and compile analytical views
db.load_csv_data(get_project_root() / "data" / "generated")
print("DuckDB Analytical Warehouse Initialized Successfully.")

DuckDB Analytical Warehouse Initialized Successfully.


---

## 3. DuckDB Vectorized Query Execution Plan & Profiling

DuckDB uses a vectorized columnar query execution engine that processes data in 2048-tuple vectors using SIMD CPU instructions. We inspect the query plan to verify how columnar scans and hash aggregates optimize financial queries:

In [2]:
# Inspect Query Execution Plan for Cost Aggregations
explain_df = db.query_df("""
    EXPLAIN ANALYZE
    SELECT cost_category, SUM(amount) as total_spend
    FROM cost_records
    GROUP BY cost_category
""")
for line in explain_df["explain_value"]:
    print(line)

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
     EXPLAIN ANALYZE     SELECT cost_category, SUM(amount) as total_spend     FROM cost_records     GROUP BY cost_category 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0007s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│                           │
│           0 rows          │
│           0.00s           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│         PROJECTION        │
│    ────────────────────   │
│__internal_decompress_strin│
│

---

## 4. Entity Table Counts & Managed Database Schema
We inspect the row counts across all managed tables:

In [3]:
entity_counts = db.get_entity_counts()
counts_df = pd.DataFrame(list(entity_counts.items()), columns=["Table Name", "Row Count"])
display(counts_df)

---

## 5. Formal Mathematical Reconciliation Proof

A mandatory requirement in IT financial governance is reconciliation: the total spend must perfectly balance across all dimensional axes:
$$\sum \text{Cost Pools} \equiv \sum \text{Categories} \equiv \sum \text{Services} \equiv \sum \text{Monthly Spend}$$

In [4]:
total_spend = db.get_total_cost()
sum_category = db.query_df("SELECT SUM(amount) FROM cost_records").iloc[0, 0]
sum_monthly = db.query_df("SELECT SUM(total_cost) FROM (SELECT SUM(amount) as total_cost FROM cost_records GROUP BY month)").iloc[0, 0]
sum_pools = db.query_df("SELECT SUM(amount) FROM (SELECT SUM(amount) as amount FROM cost_records GROUP BY cost_pool)").iloc[0, 0]

reconciliation_df = pd.DataFrame([
    {"Dimension": "Gross Ledger Sum", "Total Spend (₹)": total_spend, "Reconciled": True},
    {"Dimension": "Cost Category Sum", "Total Spend (₹)": sum_category, "Reconciled": abs(sum_category - total_spend) < 1e-4},
    {"Dimension": "Monthly Aggregation Sum", "Total Spend (₹)": sum_monthly, "Reconciled": abs(sum_monthly - total_spend) < 1e-4},
    {"Dimension": "Cost Pool Sum", "Total Spend (₹)": sum_pools, "Reconciled": abs(sum_pools - total_spend) < 1e-4},
])
display(reconciliation_df)
assert reconciliation_df["Reconciled"].all(), "Financial reconciliation failed!"

---

## 6. Multi-Dimensional Spend Breakdowns & Visualizations

We analyze spend across cost categories, IT services, applications, and business units:

In [5]:
# Spend by Category and Service
cat_df = db.get_cost_by_category()
srv_cost_df = db.get_cost_by_service()
bu_cost_df = db.get_cost_by_business_unit()

fig, axs = plt.subplots(1, 2, figsize=(13, 5))

axs[0].barh(cat_df["cost_category"], cat_df["total_cost"] / 1e7, color="#2b5c8f")
axs[0].set_title("Technology Spend by Cost Category (₹ Cr)", fontsize=11)
axs[0].set_xlabel("Spend (₹ Crores)")
axs[0].grid(axis="x", linestyle="--", alpha=0.5)

axs[1].barh(bu_cost_df.head(6)["business_unit_name"], bu_cost_df.head(6)["total_cost"] / 1e7, color="#e07a5f")
axs[1].set_title("Top 6 Business Units by Technology Spend (₹ Cr)", fontsize=11)
axs[1].set_xlabel("Spend (₹ Crores)")
axs[1].grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

In [6]:
# Interactive Plotly Sunburst: Taxonomy Hierarchy
# Cost Pool -> Cost Category -> Top Services
sunburst_query = """
    SELECT cost_pool, cost_category, s.service_name, SUM(c.amount) as spend
    FROM cost_records c
    JOIN it_services s ON c.service_id = s.service_id
    GROUP BY cost_pool, cost_category, s.service_name
"""
sb_df = db.query_df(sunburst_query)

fig = px.sunburst(
    sb_df,
    path=["cost_pool", "cost_category", "service_name"],
    values="spend",
    title="TBM Taxonomy Hierarchy: Cost Pool -> Cost Category -> IT Service",
    color_continuous_scale="Blues"
)
fig.update_layout(margin=dict(t=40, l=0, r=0, b=0))
fig.show()

In [7]:
# Monthly Technology Spend Trajectory
trend_df = db.get_monthly_cost_trend()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trend_df["month"],
    y=trend_df["total_cost"] / 1e7,
    mode="lines+markers",
    name="Monthly Spend (₹ Cr)",
    line=dict(color="#1f77b4", width=3),
    marker=dict(size=8)
))
fig.update_layout(
    title="Monthly Technology Spend Trajectory (FY2024)",
    xaxis_title="Month",
    yaxis_title="Monthly Spend (₹ Crores)",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()

---

## 7. Active Development (v2): High-Performance Columnar Architecture

In the **v2 Architecture**, we upgrade analytical storage with enterprise warehouse capabilities:
1. **Snappy-Compressed Parquet Export:** Decouples DuckDB tables into portable, cloud-native columnar files.
2. **Zero-Copy Apache Arrow Transfer:** Directly queries analytics results into in-memory Arrow tables without Python serialization overhead.
3. **Incremental Append Ingestion:** Supports real-time micro-batch telemetry ingestion with idempotent deduplication.
4. **Automated Vectorized Query Benchmarking:** Evaluates DuckDB execution latency (ms) and throughput across aggregations and window joins.
5. **Schema Version Control & Migrations:** Tracks DDL migrations in `_schema_migrations` table for auditable warehouse evolution.

In [8]:
# 1. Export Warehouse to Snappy-Compressed Parquet
parquet_files = db.export_to_parquet()
print(f"Exported {len(parquet_files)} tables to Parquet storage:")
for p in parquet_files[:4]:
    print(f"  • {Path(p).name}")

# 2. Execute Zero-Copy Apache Arrow Query
arrow_table = db.query_arrow("""
    SELECT application_id, application_name, annual_license_cost 
    FROM applications 
    WHERE annual_license_cost > 10000000 
    ORDER BY annual_license_cost DESC
""")
print(f"\nArrow Table Schema & Record Count: {len(arrow_table)} rows, {len(arrow_table.column_names)} columns (Zero-Copy In-Memory)")
display(arrow_table.to_pandas().head())

# 3. Micro-Batch Streaming Telemetry Ingestion with Idempotent Deduplication
batch_records = [{
    "consumption_id": "CON_STREAM_001",
    "month": "2024-12",
    "application_id": "APP001",
    "business_unit_id": "BU001",
    "active_users": 8600,
    "transactions": 6650000,
    "api_calls": 16500000,
    "compute_hours": 1550.0,
    "storage_gb": 4600.0,
    "tickets": 22,
}]
inserted_count = db.ingest_telemetry_batch(batch_records, deduplicate=True)
print(f"\nStreaming Telemetry Batch Ingestion: {inserted_count} new record(s) appended")

# 4. Multi-Iteration Vectorized Query Benchmarking Suite
benchmarks = db.benchmark_queries(iterations=5)
bench_df = pd.DataFrame(benchmarks).T[["avg_ms", "min_ms", "max_ms", "std_ms", "result_rows"]]
bench_df.columns = ["Avg Latency (ms)", "Min (ms)", "Max (ms)", "Std Dev (ms)", "Result Rows"]
print("\nDuckDB Vectorized Analytical Query Benchmark:")
display(bench_df)

# 5. Schema Migration & Version Control Tooling
applied_migrations = db.apply_migrations()
migration_history = db.query_df("SELECT * FROM _schema_migrations ORDER BY applied_at ASC")
print(f"\nSchema Migrations Status: {len(applied_migrations)} newly applied, {len(migration_history)} total recorded")
display(migration_history)

Exported 20 tables to Parquet storage:
  • applications.parquet
  • benefits.parquet
  • business_capabilities.parquet
  • business_units.parquet

Arrow Table Schema & Record Count: 34 rows, 3 columns (Zero-Copy In-Memory)

Streaming Telemetry Batch Ingestion: 1 new record(s) appended

DuckDB Vectorized Analytical Query Benchmark:

Schema Migrations Status: 0 newly applied, 3 total recorded


## 7. Limitations & Analytical Guardrails

1. **Taxonomy Disclaimer:** This model is a **simplified TBM/ITFM-inspired demonstration model**. It is not an exact implementation of proprietary vendor taxonomies.
2. **Relational Limitation:** While DuckDB delivers instant columnar aggregations, it cannot natively answer multi-hop relationship queries (e.g. tracing third-party software risks to downstream customer capabilities).

## 8. Next Step

In **Notebook 03: Build Knowledge Graph**, we transform this relational data into a typed property graph using **NetworkX**, creating the relationship layer that powers multi-hop dependency and capability traversal.